# 21 Run Isolation Check

Purpose: verify that v1/v2 runs will not mix shared feature or dataset artifacts. Use this before increasing the video tranche or creating a v2 profile.

Runtime: CPU.

This notebook reads Drive artifacts and prints an audit. It does not modify Drive unless `WRITE_NEXT_PROFILE = True`.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


os.environ.setdefault('BATTING_CODE_ROOT', '/content/drive/MyDrive/codex/batting_codex_handoff')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path(os.environ['BATTING_CODE_ROOT'])
    mydrive = Path('/content/drive/MyDrive')
    candidates = [
        fixed,
        Path('/content/drive/MyDrive/codex/batting_codex_handoff'),
        Path('/content/drive/MyDrive/batting_codex_handoff'),
        Path.cwd(),
        *Path.cwd().parents,
    ]
    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            return candidate
    raise FileNotFoundError(
        'sport_pipeline repo が見つかりません。Drive mount と repo 配置を確認してください.\n'
        f'BATTING_CODE_ROOT={fixed}\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別配置の場合は notebook の最初の cell より前に次を実行してください.\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/あなたの配置/batting_codex_handoff\n'
        f'MyDrive exists={mydrive.exists()}\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()
REPO_DIR = _resolve_repo_dir()
os.environ['BATTING_CODE_ROOT'] = str(REPO_DIR)
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME
src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from sport_pipeline.pipeline.run_isolation import audit_run_isolation, make_next_run_profile
from sport_pipeline.pipeline.run_profile import load_run_profile

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
audit = audit_run_isolation(RUN_PROFILE, base_dir=BASE_DIR)
print(json.dumps(audit, ensure_ascii=False, indent=2))


In [ ]:
# Optional v2 profile preview. This cell does not write unless WRITE_NEXT_PROFILE is True.
NEXT_VERSION = 'v2'
NEXT_MAX_FILES = 1500
NEXT_MAX_CLIPS = 1500
WRITE_NEXT_PROFILE = False

next_profile = make_next_run_profile(
    RUN_PROFILE,
    version=NEXT_VERSION,
    max_files=NEXT_MAX_FILES,
    max_clips=NEXT_MAX_CLIPS,
)
next_audit = audit_run_isolation(next_profile, base_dir=BASE_DIR)
print(json.dumps(next_audit, ensure_ascii=False, indent=2))

if WRITE_NEXT_PROFILE:
    output_path = REPO_DIR / 'configs' / 'runs' / f'mlb_2024_2026_real_colab_{NEXT_VERSION}.json'
    output_path.write_text(json.dumps(next_profile, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('wrote', output_path)
else:
    print('preview only; set WRITE_NEXT_PROFILE=True when you intentionally want to create the profile file')
